# 5.3 Evaluation

This notebook performs the final evaluation of all three trained models against the NSL-KDD test set.

**Contents:**
1. Reload preprocessed data and retrain models
2. Per-model classification reports
3. ROC curves + AUC
4. Precision-Recall curves + Average Precision
5. Radar chart — multi-metric visual comparison
6. Final comparison table (metrics + stability)
7. Interpretation — which model wins and why

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from src.preprocess import preprocess
from src.train import train_knn, train_nb, train_svm, stability_analysis
from src.evaluate import (
    metrics_summary,
    plot_roc,
    plot_pr,
    final_comparison_table,
    print_classification_reports,
    plot_radar,
    interpret_results,
)

sns.set_theme(style="whitegrid", palette="muted")
REPORT_DIR = "../reports"
os.makedirs(REPORT_DIR, exist_ok=True)
print('Imports OK')

Imports OK


## Step 1 — Load Data

Load from `data/processed/` — already preprocessed in notebook 5.1.

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"y_train: {y_train.value_counts().to_dict()}")
print(f"y_test : {y_test.value_counts().to_dict()}")

X_train: (125973, 41)  X_test: (22543, 41)
y_train: {0: 67343, 1: 58630}
y_test : {1: 12833, 0: 9710}


## Step 2 — Retrain Models

We retrain using the same best hyperparameters found in notebook 5.2.


In [ ]:
# KNN — retrain with best K from 5.2 (change if your best_k was different)
knn_model, knn_metrics, knn_cv_scores, best_k = train_knn(
    X_train, y_train, X_test, y_test,
    k_range=range(1, 22, 2)
)

# Naive Bayes
nb_model, nb_metrics = train_nb(X_train, y_train, X_test, y_test)

# SVM
svm_model, svm_metrics, svm_best_params, svm_sample = train_svm(
    X_train, y_train, X_test, y_test,
    sample_size=25000, cv=3
)

[tune_knn] Testing K = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21] with 5-fold CV...
  K= 1  F1=0.9963 ± 0.0004
  K= 3  F1=0.9955 ± 0.0003
  K= 5  F1=0.9950 ± 0.0003
  K= 7  F1=0.9943 ± 0.0004
  K= 9  F1=0.9935 ± 0.0003
  K=11  F1=0.9930 ± 0.0006
  K=13  F1=0.9928 ± 0.0006
  K=15  F1=0.9925 ± 0.0006


## Step 3 — Classification Reports

Detailed per-class breakdown showing precision, recall, and F1 for both Normal and Attack classes separately.

In [ ]:
predictions = {
    "KNN":         knn_metrics["y_pred"],
    "Naive Bayes": nb_metrics["y_pred"],
    "SVM":         svm_metrics["y_pred"],
}

print_classification_reports(y_test, predictions)

## Step 4 — ROC Curves

ROC (Receiver Operating Characteristic) curves plot True Positive Rate vs False Positive Rate at every classification threshold. AUC (Area Under Curve) summarises performance in a single number — 1.0 = perfect, 0.5 = random.

**Note on SVM:** `SVC` with default settings does not produce probability estimates. We use `decision_function()` which returns a signed distance from the decision boundary — a valid ranking score for ROC/PR computation.

In [ ]:
# Get probability / score estimates for each model
proba_dict = {
    "KNN":         knn_model.predict_proba(X_test)[:, 1],
    "Naive Bayes": nb_model.predict_proba(X_test)[:, 1],
    "SVM":         svm_model.decision_function(X_test),   # signed distance
}

auc_scores = plot_roc(
    y_test, proba_dict,
    save_path=f"{REPORT_DIR}/roc_curves.png"
)

print("\nAUC Scores:")
for name, score in auc_scores.items():
    print(f"  {name:15s}: {score:.4f}")

## Step 5 — Precision-Recall Curves

PR curves are particularly relevant for intrusion detection because:
- **Recall** (y-axis at threshold=0.5) = fraction of actual attacks detected → our priority metric
- **Precision** = fraction of flagged connections that are truly attacks → false alarm rate

A model can achieve high Recall by flagging everything, but precision collapses. A good NIDS needs both to be high. **Average Precision (AP)** summarises the area under the PR curve.

In [ ]:
ap_scores = plot_pr(
    y_test, proba_dict,
    save_path=f"{REPORT_DIR}/pr_curves.png"
)

print("\nAverage Precision Scores:")
for name, score in ap_scores.items():
    print(f"  {name:15s}: {score:.4f}")

## Step 6 — Stability Analysis

Repeated Stratified K-Fold (10×5 = 50 evaluations per model) measures how consistent each model's F1 is across different data splits.

**CV% (Coefficient of Variation)** = std / mean × 100
- CV% < 2% → very stable
- CV% 2–5% → acceptable
- CV% > 5% → unstable

In [ ]:
stability_models = {
    "KNN":         KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "SVM":         SVC(kernel="rbf",
                       C=svm_best_params["C"],
                       gamma=svm_best_params["gamma"],
                       class_weight="balanced",
                       random_state=42),
}

stability_df, raw_scores = stability_analysis(
    stability_models, X_train, y_train,
    n_splits=5, n_repeats=10,
    scoring="f1", svm_sample_size=25000
)

print("\nStability Summary:")
print(stability_df.set_index("Model").to_string())

## Step 7 — Radar Chart

In [ ]:
metrics_df = metrics_summary(y_test, predictions)

plot_radar(
    metrics_df,
    save_path=f"{REPORT_DIR}/radar_chart.png"
)

## Step 8 — Final Comparison Table

In [ ]:
table = final_comparison_table(
    metrics_df, stability_df, auc_scores, ap_scores
)

print("Final Comparison Table:")
print(table.round(4).to_string())

# Save as CSV for the report
table.round(4).to_csv(f"{REPORT_DIR}/final_comparison_table.csv")
print(f"\nSaved → {REPORT_DIR}/final_comparison_table.csv")

In [ ]:
# Heatmap version of the table (great for the report)
plot_cols = ["Accuracy", "Precision", "Recall", "F1", "AUC", "Avg Precision"]

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(
    table[plot_cols].astype(float),
    annot=True, fmt=".4f", cmap="YlGnBu",
    linewidths=0.5, ax=ax,
    vmin=0.85, vmax=1.0
)
ax.set_title("Final Model Comparison — Metric Heatmap")
plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/final_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9 — Interpretation

In [ ]:
interpret_results(table)

---
## Summary of All Outputs

In [ ]:
all_reports = [
    "feature_groups.png",
    "class_balance.png",
    "scaling_example.png",
    "knn_k_selection.png",
    "model_comparison.png",
    "confusion_matrices.png",
    "stability_boxplot.png",
    "learning_curves.png",
    "roc_curves.png",
    "pr_curves.png",
    "radar_chart.png",
    "final_heatmap.png",
    "final_comparison_table.csv",
]

print("=" * 50)
print("  PROJECT COMPLETE — All outputs in reports/")
print("=" * 50)
for f in all_reports:
    path = f"{REPORT_DIR}/{f}"
    status = "✅" if os.path.exists(path) else "❌ missing"
    print(f"  {status}  {f}")
print("=" * 50)